# Fine-tune Qwen2.5-Coder and infer on the TypePro test split

Settings: **Internet ON**, accelerator **GPU**. Attach the final private
dataset `typepro-python-generative` using **Add Input**.


In [ ]:
REPOSITORY = 'https://github.com/duyvu1105/TypePro.git'
BRANCH = 'main'
MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
INPUT_LENGTH = 16384
LABEL_LENGTH = 128
EPOCHS = 3

import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/TypePro")
OUTPUT_DIR = Path("/kaggle/working/codet5p-typepro-python")

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, REPO_DIR])
else:
    # A rerun can reuse /kaggle/working/TypePro; refresh it so helper
    # modules such as generative_chat.py match this notebook version.
    run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])
PIPELINE_DIR = REPO_DIR / "codet5p_type_retrieval"
run([sys.executable, "-m", "pip", "install", "-q", "-r", PIPELINE_DIR / "requirements.txt"])


## Locate and verify the attached processed dataset


In [ ]:
candidates = []
for path in Path("/kaggle/input").rglob("manifest.json"):
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if value.get("schema_version") == "typepro-codet5p-generative-project-kb-v2":
        candidates.append(path.parent)
if len(candidates) != 1:
    raise RuntimeError(f"Expected exactly one TypePro processed dataset, found {candidates}")
DATA_DIR = candidates[0]
print("Using dataset:", DATA_DIR)
run([sys.executable, PIPELINE_DIR / "verify_dataset.py", "--data-dir", DATA_DIR])


## Measure token lengths before training


In [ ]:
from transformers import AutoTokenizer
pipeline_path = str(PIPELINE_DIR)
if pipeline_path not in sys.path:
    sys.path.insert(0, pipeline_path)
from generative_chat import chat_token_ids

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Measurement intentionally keeps long sequences untruncated.
tokenizer.model_max_length = 10**9

def empty_token_stats(limit):
    return {
        "samples": 0,
        "total_tokens": 0,
        "min_tokens": None,
        "max_tokens": 0,
        "over_limit": 0,
        "configured_limit": limit,
    }

def update_token_stats(stats, lengths):
    for length in lengths:
        stats["samples"] += 1
        stats["total_tokens"] += length
        stats["min_tokens"] = (
            length if stats["min_tokens"] is None
            else min(stats["min_tokens"], length)
        )
        stats["max_tokens"] = max(stats["max_tokens"], length)
        stats["over_limit"] += int(length > stats["configured_limit"])

def finalized_token_stats(stats):
    count = stats["samples"]
    return {
        "samples": count,
        "average_tokens": round(stats["total_tokens"] / count, 2) if count else 0.0,
        "min_tokens": stats["min_tokens"] or 0,
        "max_tokens": stats["max_tokens"],
        "configured_limit": stats["configured_limit"],
        "over_limit": stats["over_limit"],
        "over_limit_percentage": (
            round(100.0 * stats["over_limit"] / count, 2) if count else 0.0
        ),
    }

def measure_split(path, batch_size=256):
    input_stats = empty_token_stats(INPUT_LENGTH)
    label_stats = empty_token_stats(LABEL_LENGTH)
    batch = []

    def measure_batch(rows):
        if not rows:
            return
        input_ids = [chat_token_ids(tokenizer, row["input"]) for row in rows]
        label_ids = []
        for row, prompt_ids in zip(rows, input_ids):
            full_ids = chat_token_ids(tokenizer, row["input"], row["label"])
            if full_ids[:len(prompt_ids)] != prompt_ids:
                raise ValueError("Chat template did not preserve the prompt as a sequence prefix")
            label_ids.append(full_ids[len(prompt_ids):])
        update_token_stats(input_stats, [len(ids) for ids in input_ids])
        update_token_stats(label_stats, [len(ids) for ids in label_ids])

    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            batch.append(json.loads(line))
            if len(batch) >= batch_size:
                measure_batch(batch)
                batch.clear()
        measure_batch(batch)
    return {"input": input_stats, "label": label_stats}

raw_token_stats = {}
overall = {
    "input": empty_token_stats(INPUT_LENGTH),
    "label": empty_token_stats(LABEL_LENGTH),
}
for split in ("train", "validation", "test"):
    raw_token_stats[split] = measure_split(DATA_DIR / f"{split}.jsonl")
    print(json.dumps({
        "split": split,
        "input": finalized_token_stats(raw_token_stats[split]["input"]),
        "label": finalized_token_stats(raw_token_stats[split]["label"]),
    }, indent=2, ensure_ascii=False), flush=True)
    for field in ("input", "label"):
        current = raw_token_stats[split][field]
        combined = overall[field]
        combined["samples"] += current["samples"]
        combined["total_tokens"] += current["total_tokens"]
        combined["max_tokens"] = max(combined["max_tokens"], current["max_tokens"])
        if current["min_tokens"] is not None:
            combined["min_tokens"] = (
                current["min_tokens"] if combined["min_tokens"] is None
                else min(combined["min_tokens"], current["min_tokens"])
            )
        combined["over_limit"] += current["over_limit"]

token_stats = {
    split: {
        field: finalized_token_stats(values)
        for field, values in split_values.items()
    }
    for split, split_values in raw_token_stats.items()
}
token_stats["overall"] = {
    field: finalized_token_stats(values)
    for field, values in overall.items()
}
print("===== TOKEN LENGTH STATISTICS (NO TRUNCATION) =====")
print(json.dumps(token_stats, indent=2, ensure_ascii=False))


## Generative sequence-to-sequence fine-tuning


In [ ]:
run([
    "accelerate", "launch", "--multi_gpu",
    PIPELINE_DIR / "train_generative.py",
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    "--model-name", MODEL_NAME,
    "--input-length", INPUT_LENGTH,
    "--label-length", LABEL_LENGTH,
    "--batch-size", 1,
    "--gradient-accumulation-steps", 16,
    "--epochs", EPOCHS,
    "--learning-rate", "2e-5",
    "--mixed-precision", "fp16",
    "--gradient-checkpointing",
    "--seed", 13,
])


## Batched inference and test metrics


In [ ]:
predictions = Path("/kaggle/working/test_predictions.jsonl")
run([
    sys.executable, "-u", PIPELINE_DIR / "infer_generative.py",
    "--checkpoint", OUTPUT_DIR / "best",
    "--model-name", MODEL_NAME,
    "--input", DATA_DIR / "test.jsonl",
    "--output", predictions,
    "--input-length", INPUT_LENGTH,
    "--label-length", LABEL_LENGTH,
    "--batch-size", 4,
])
print("Predictions:", predictions)
